In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
 
df = pd.read_parquet(r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\model_df.parquet")
df_raw = pd.read_parquet(r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\combined_df_finalcleaned.parquet")

In [7]:
print(f"Total columns: {len(df.columns)}")
print()
for i, c in enumerate(df.columns):
    print(f"  {i+1:3}. {c:60} {str(df[c].dtype)}")

Total columns: 186

    1. metadata_price                                               float64
    2. question_recommendation_nps_a                                float64
    3. question_overall_satisfaction_booking_experience             Int8
    4. question_overall_satisfaction_wifi_onboard_the_train         Int8
    5. question_overall_satisfaction_experience_at_departure_station Int8
    6. question_overall_satisfaction_journey_punctuality            Int8
    7. question_overall_satisfaction_comfort_onboard_the_train      Int8
    8. question_overall_satisfaction_cleanliness_onboard_the_train  Int8
    9. question_overall_satisfaction_overall_service_from_eurostar_staff Int8
   10. question_overall_satisfaction_information_provided_to_you_before_travelling Int8
   11. question_at_the_station_checking_in_to_departures_area       Int8
   12. question_at_the_station_boarding_the_train                   Int8
   13. question_at_the_station_toilets_washrooms_in_the_station     Int8
   1

In [6]:
print(f"Total columns: {len(df_raw.columns)}")
print()
for i, c in enumerate(df_raw.columns):
    print(f"  {i+1:3}. {c:60} {str(df_raw[c].dtype)}")

Total columns: 274

    1. metadata_booking_date                                        str
    2. metadata_class_of_service                                    str
    3. metadata_class_of_service_code                               float64
    4. metadata_coach_number                                        int64
    5. metadata_compensation_flag                                   int64
    6. metadata_currency                                            str
    7. metadata_delay_code                                          float64
    8. metadata_delay_at_arrival                                    float64
    9. metadata_departure_date                                      int64
   10. metadata_departure_hour                                      str
   11. metadata_destination_station                                 str
   12. metadata_destination_station_code                            float64
   13. metadata_disrup                                              str
   14. metadata_origin

In [2]:
target = "question_recommendation_nps_a"
print(f"Shape: {df.shape}")
print(f"Target range: {df[target].min():.0f} - {df[target].max():.0f}")
print(f"Target mean: {df[target].mean():.2f}")
print(f"Total nulls: {df.isnull().sum().sum()}")

Shape: (178384, 186)
Target range: 0 - 10
Target mean: 8.07
Total nulls: 6188971


In [3]:
null_summary = (
    df.isnull().sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "null_count"})
)
null_summary["null_pct"] = (null_summary["null_count"] / len(df) * 100).round(1)
null_summary = null_summary[null_summary["null_count"] > 0].sort_values("null_pct", ascending=False)
 
print(f"Columns with nulls: {len(null_summary)} of {df.shape[1]}")
print("\n--- >50% null ---")
print(null_summary[null_summary["null_pct"] > 50].to_string(index=False))
print("\n--- 10-50% null ---")
print(null_summary[(null_summary["null_pct"] >= 10) & (null_summary["null_pct"] <= 50)].to_string(index=False))
print("\n--- <10% null ---")
print(null_summary[null_summary["null_pct"] < 10].to_string(index=False))

Columns with nulls: 69 of 186

--- >50% null ---
                                                                                      column  null_count  null_pct
                                                   question_on_board_the_train1_seat_comfort      176211      98.8
question_arrival_the_goodbye_you_received_from_train_staff_when_you_reached_your_destination      176327      98.8
                                    question_service_provided_by_at_the_eurostar_caf_onboard      169156      94.8
              question_on_board_the_train2_range_of_food_drink_available_at_the_eurostar_caf      167919      94.1
                           question_arrival_onboard_announcement_on_arrival_by_train_manager      163224      91.5
                   question_service_provided_by_when_getting_in_touch_via_the_contact_centre      160804      90.1
                                  question_at_the_station_the_facilities_in_the_waiting_area      150694      84.5
                         questi

In [4]:
high_null_cols = null_summary[null_summary["null_pct"] > 75]["column"].tolist()
print(f"Columns >75% null: {len(high_null_cols)}")
 
correlations = df[high_null_cols + [target]].corr()[target].drop(target).abs().sort_values(ascending=False)
print(f"Max NPS correlation among these: {correlations.max():.3f}")
print(f"Mean NPS correlation: {correlations.mean():.3f}")
 
df_model = df.drop(columns=high_null_cols)
print(f"Columns after dropping >75% null: {df_model.shape[1]} | rows unchanged: {df_model.shape[0]}")

Columns >75% null: 28
Max NPS correlation among these: 0.492
Mean NPS correlation: 0.403
Columns after dropping >75% null: 158 | rows unchanged: 178384


In [5]:
remaining_nulls = df_model.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print(f"Remaining columns with nulls: {len(remaining_nulls)}")
print(remaining_nulls.to_string())
 
straggler_cols = [
    "question_on_board_the_train1_the_noise_on_board",
    "question_on_board_the_train1_the_temperature_on_board",
]
df_model = df_model.drop(columns=straggler_cols)
print(f"Columns after straggler drop: {df_model.shape[1]}")

Remaining columns with nulls: 41
question_on_board_the_train1_the_noise_on_board                                                  133662
question_on_board_the_train1_the_temperature_on_board                                            133561
pm_days_since_comms                                                                               88565
clean_score_deep                                                                                  85663
clean_days_since_deep                                                                             85663
pax_quartile_check                                                                                69697
pct_youth                                                                                         69697
pct_adult                                                                                         69697
has_equipment_change                                                                              69697
pct_senior                     

In [6]:
decimal_cols = ["compensation_liability_evouchers", "compensation_liability_cash"]
df_model[decimal_cols] = df_model[decimal_cols].astype(float)
print(df_model[decimal_cols].dtypes)

compensation_liability_evouchers    float64
compensation_liability_cash         float64
dtype: object


In [7]:
pm_days_cols = [c for c in df_model.columns if "pm_days_since" in c]
print(f"{'column':<35}{'rho_is_channel':>16}{'rho_NPS':>10}{'null_pct':>10}")
for col in pm_days_cols:
    r_channel = df_model[col].corr(df_model["is_channel"])
    r_nps = df_model[col].corr(df_model[target])
    null_pct = df_model[col].isnull().mean() * 100
    print(f"{col:<35}{r_channel:>16.3f}{r_nps:>10.3f}{null_pct:>9.1f}%")
 

column                               rho_is_channel   rho_NPS  null_pct
pm_days_since_catering                        0.004     0.032     33.7%
pm_days_since_toilet                          0.051     0.007     35.3%
pm_days_since_climate                         0.045     0.003     33.7%
pm_days_since_interior                        0.018    -0.007     33.7%
pm_days_since_comms                          -0.041    -0.012     49.6%
pm_days_since_catering_resid                 -0.000     0.025      0.0%
pm_days_since_catering_has_data               0.976     0.104      0.0%
pm_days_since_toilet_resid                    0.000     0.004      0.0%
pm_days_since_toilet_has_data                 0.943     0.101      0.0%
pm_days_since_climate_resid                  -0.000     0.002      0.0%
pm_days_since_climate_has_data                0.976     0.104      0.0%
pm_days_since_interior_resid                 -0.000    -0.006      0.0%
pm_days_since_interior_has_data               0.976     0.104   

In [8]:
cm_open_cols = [c for c in df_model.columns if "cm_open" in c]
print(f"{'column':<35}{'rho_is_channel':>16}{'rho_NPS':>10}{'null_pct':>10}")
for col in cm_open_cols:
    r_channel = df_model[col].corr(df_model["is_channel"])
    r_nps = df_model[col].corr(df_model[target])
    null_pct = df_model[col].isnull().mean() * 100
    print(f"{col:<35}{r_channel:>16.3f}{r_nps:>10.3f}{null_pct:>9.1f}%")

column                               rho_is_channel   rho_NPS  null_pct
cm_open_climate                              -0.767    -0.084      0.0%
cm_open_wifi                                 -0.217    -0.017      0.0%
cm_open_comms                                -0.445    -0.046      0.0%
cm_open_interior                             -0.672    -0.065      0.0%
cm_open_catering                             -0.735    -0.070      0.0%
cm_open_toilet                               -0.789    -0.086      0.0%
cm_open_cleaning                             -0.704    -0.072      0.0%
cm_open_climate_resid                         0.000    -0.003      0.0%
cm_open_toilet_resid                          0.000    -0.003      0.0%
cm_open_interior_resid                        0.000     0.009      0.0%
cm_open_catering_resid                        0.000     0.013      0.0%
cm_open_cleaning_resid                       -0.000     0.005      0.0%
cm_open_wifi_resid                            0.000     0.007   

In [9]:
channel = df_model[df_model["is_channel"] == 1]
print(channel["equipment_type"].value_counts())
print(channel.groupby("equipment_type")[["pm_days_since_comms", "clean_score_deep", "clean_days_since_deep"]]
      .apply(lambda g: g.isnull().mean()))

equipment_type
E320    90847
E300    25418
Name: count, dtype: int64
                pm_days_since_comms  clean_score_deep  clean_days_since_deep
equipment_type                                                              
E300                       0.062043          0.293296               0.293296
E320                       0.288925          0.190782               0.190782


In [10]:
print(df_model["total_open_faults"].describe())
print(f"Zero values: {(df_model['total_open_faults']==0).sum():,} ({(df_model['total_open_faults']==0).mean()*100:.1f}%)")
print(f"Nulls: {df_model['total_open_faults'].isnull().sum()}")
print(df_model.groupby("is_channel")["total_open_faults"].describe().round(2))
print(f"rho_is_channel: {df_model['total_open_faults'].corr(df_model['is_channel']):.4f}")
print(f"rho_NPS: {df_model['total_open_faults'].corr(df_model[target]):.4f}")
 
cm_cols = [c for c in df_model.columns if c.startswith("cm_open_") and not c.endswith("_resid")]
print("Consistency check (sum of cm_open_* vs total_open_faults), max abs diff:",
      (df_model[cm_cols].sum(axis=1) - df_model["total_open_faults"]).abs().max())
 
print(df_model.groupby("is_channel")["total_open_faults"].agg(["mean", "median", "max", lambda x: (x == 0).mean()]))

count    178384.000000
mean         16.360397
std          25.521461
min           0.000000
25%           0.000000
50%           0.000000
75%          35.000000
max         136.000000
Name: total_open_faults, dtype: float64
Zero values: 118,192 (66.3%)
Nulls: 0
               count   mean    std  min   25%   50%   75%    max
is_channel                                                      
0            62119.0  46.98  20.78  0.0  34.0  44.0  55.0  136.0
1           116265.0   0.00   0.00  0.0   0.0   0.0   0.0    0.0
rho_is_channel: -0.8770
rho_NPS: -0.0890
Consistency check (sum of cm_open_* vs total_open_faults), max abs diff: 0.0
                 mean  median    max  <lambda_0>
is_channel                                      
0           46.981326    44.0  136.0    0.031021
1            0.000000     0.0    0.0    1.000000


In [11]:
null_cols = df_model.columns[df_model.isnull().any()]
audit_null = pd.DataFrame({
    "null_pct": df_model[null_cols].isnull().mean().mul(100).round(1),
    "rho_is_channel": [df_model[c].isnull().astype(int).corr(df_model["is_channel"]) for c in null_cols],
}).round(3).sort_values("null_pct", ascending=False)
print(audit_null.to_string())
 
# %%
both_drop = ["future_loyalty", "clean_has_prior_routine", "clean_has_prior_deep"]
df_full = df_model.drop(columns=both_drop)
print(f"After dropping confirmed-bad columns: {df_full.shape}")

                                                                                               null_pct  rho_is_channel
pm_days_since_comms                                                                                49.6          -0.704
clean_days_since_deep                                                                              48.0          -0.731
clean_score_deep                                                                                   48.0          -0.731
pct_child                                                                                          39.1          -0.913
pct_youth                                                                                          39.1          -0.913
pct_adult                                                                                          39.1          -0.913
has_equipment_change                                                                               39.1          -0.913
pct_senior                              

In [12]:
print("total_pax nulls:", df_full["total_pax"].isnull().sum(), "/", len(df_full))
print(df_full.groupby("is_channel")["total_pax"].apply(lambda x: x.isnull().mean()).rename("null_rate"))
print("\nAmong rows where total_pax IS observed, share <=27:",
      df_full.loc[df_full["total_pax"].notnull(), "total_pax"].le(27).mean().round(3))
print("\nhas_equipment_change nulls:", df_full["has_equipment_change"].isnull().sum())
print(df_full.groupby("is_channel")["has_equipment_change"].apply(lambda x: x.isnull().mean()))

total_pax nulls: 69697 / 178384
is_channel
0    1.000000
1    0.065179
Name: null_rate, dtype: float64

Among rows where total_pax IS observed, share <=27: 0.0

has_equipment_change nulls: 69697
is_channel
0    1.000000
1    0.065179
Name: has_equipment_change, dtype: float64


In [13]:
bucket1_zero_fill = ["clean_score_routine", "clean_score_deep"]

bucket2_median_fill = [
    "clean_hours_since_routine", "clean_days_since_deep",
    "clean_recency_encoded",  # verify direction - ordinal recency, not a count
    "pm_days_since_comms", "pm_days_since_toilet", "pm_days_since_climate",
    "pm_days_since_interior", "pm_days_since_catering",
    "average_days_since_last_exams",
]

In [14]:
bucket3_group = [
    "pct_child", "pct_youth", "pct_adult", "pct_senior",
    "avg_group_size", "group_pax_ratio", "total_pax", "pax_quartile_check",
    "has_equipment_change",
]

In [15]:
all_bucket_cols = bucket1_zero_fill + bucket1_one_fill + bucket2_median_fill + bucket3_group
print(df_full[all_bucket_cols].isnull().sum().sort_values(ascending=False))
print("\nRows touched by bucket 3:", df_full[bucket3_group].isnull().any(axis=1).sum())
print("Rows touched by bucket 1+2:", df_full[bucket1_zero_fill + bucket1_one_fill + bucket2_median_fill].isnull().any(axis=1).sum())
print("Overlap:", (df_full[bucket3_group].isnull().any(axis=1) &
                    df_full[bucket1_zero_fill + bucket1_one_fill + bucket2_median_fill].isnull().any(axis=1)).sum())
print("\nTotal unique rows touched by any bucketed fill (of", len(df_full), "):",
      df_full[all_bucket_cols].isnull().any(axis=1).sum())

pm_days_since_comms              88565
clean_days_since_deep            85663
clean_score_deep                 85663
pct_child                        69697
total_pax                        69697
pax_quartile_check               69697
has_equipment_change             69697
pct_adult                        69697
group_pax_ratio                  69697
avg_group_size                   69697
pct_senior                       69697
pct_youth                        69697
pm_days_since_toilet             62955
clean_hours_since_routine        62164
clean_below_90                   62164
clean_score_routine              62164
clean_recency_encoded            62164
average_days_since_last_exams    60192
pm_days_since_catering           60192
pm_days_since_interior           60192
pm_days_since_climate            60192
dtype: int64

Rows touched by bucket 3: 69697
Rows touched by bucket 1+2: 113492
Overlap: 63316

Total unique rows touched by any bucketed fill (of 178384 ): 119873


In [16]:

for col in bucket1_zero_fill:
    df_full[f"{col}_was_tracked"] = df_full[col].notnull().astype(int)
    df_full[col] = df_full[col].fillna(0)
 
for col in bucket1_one_fill:
    df_full[f"{col}_was_tracked"] = df_full[col].notnull().astype(int)
    df_full[col] = df_full[col].fillna(1)
 
print("Bucket 1 nulls remaining:", df_full[bucket1_zero_fill + bucket1_one_fill].isnull().sum().sum())

Bucket 1 nulls remaining: 0


In [17]:

for col in bucket2_median_fill:
    df_full[f"{col}_was_tracked"] = df_full[col].notnull().astype(int)
    df_full[col] = df_full[col].astype("float64")  # group medians can be fractional; nullable Int dtypes (e.g. clean_recency_encoded as Int8) reject that on fillna otherwise
    group_medians = df_full.groupby("equipment_type")[col].median()
    df_full[col] = df_full[col].fillna(df_full["equipment_type"].map(group_medians))
    df_full[col] = df_full[col].fillna(df_full[col].median())  # backstop if an equip type has zero real values

In [18]:
print("Bucket 2 nulls remaining:", df_full[bucket2_median_fill].isnull().sum().sum())
 


Bucket 2 nulls remaining: 0


In [19]:
df_full["group_data_available"] = df_full["total_pax"].notnull().astype(int)
df_full["has_equipment_change"] = df_full["has_equipment_change"].fillna(df_full["has_equipment_change"].mode()[0])
 
pax_quartile_order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
df_full["pax_quartile_check"] = df_full["pax_quartile_check"].map(pax_quartile_order)
df_full["pax_quartile_check"] = df_full["pax_quartile_check"].fillna(df_full["pax_quartile_check"].mode()[0])
 
continuous_bucket3 = [c for c in bucket3_group if c not in ("has_equipment_change", "pax_quartile_check")]
for col in continuous_bucket3:
    df_full[col] = df_full[col].fillna(df_full[col].median())
 
print("Bucket 3 nulls remaining:", df_full[bucket3_group].isnull().sum().sum())

Bucket 3 nulls remaining: 0


In [20]:
remaining_null_cols = df_full.columns[df_full.isnull().any()].tolist()
print("Plain median fill for:", remaining_null_cols)
for col in remaining_null_cols:
    df_full[col] = df_full[col].fillna(df_full[col].median())
 
print("Total nulls left in df_full:", df_full.isnull().sum().sum())

Plain median fill for: ['question_overall_satisfaction_booking_experience', 'question_overall_satisfaction_wifi_onboard_the_train', 'question_overall_satisfaction_experience_at_departure_station', 'question_overall_satisfaction_journey_punctuality', 'question_overall_satisfaction_comfort_onboard_the_train', 'question_overall_satisfaction_cleanliness_onboard_the_train', 'question_overall_satisfaction_overall_service_from_eurostar_staff', 'question_overall_satisfaction_information_provided_to_you_before_travelling', 'question_staff_manners_staff_were_visible', 'question_staff_manners_staff_were_friendly_and_welcoming', 'question_staff_manners_staff_were_helpful', 'question_ticket_price_satisfa', 'question_strategic_pillar_ass_travelling_by_eurostar_is_an_environmentally_sustainable_option', 'last_clean_score', 'hours_since_last_clean', 'pm_has_prior_wifi', 'staff_composite', 'pm_has_prior_wifi_resid']
Total nulls left in df_full: 0


In [21]:

equip_dummies = pd.get_dummies(df_full["equipment_type"], prefix="equip")
equip_dummies = equip_dummies.drop(columns=["equip_E300"])
df_full = pd.concat([df_full.drop(columns=["equipment_type"]), equip_dummies], axis=1)
print(df_full.shape)
print(df_full.filter(like="equip_").sum())
 

(178384, 168)
equip_E320    92674
equip_RUB      6214
equip_TGH     53984
dtype: int64


In [22]:
lean_drop = [
    "pm_days_since_comms", "pm_days_since_toilet",
    "pm_days_since_climate", "pm_days_since_interior", "pm_days_since_catering",
    "clean_score_deep", "clean_days_since_deep",
]
lean_drop_full = lean_drop + [f"{c}_was_tracked" for c in lean_drop if f"{c}_was_tracked" in df_full.columns]
df_lean = df_full.drop(columns=lean_drop_full)
print(f"df_full: {df_full.shape}  df_lean: {df_lean.shape}")
 

df_full: (178384, 168)  df_lean: (178384, 154)


In [23]:
full_path = r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\model_df_full.parquet"
lean_path = r"C:\Users\gehan\Downloads\thesis\extracted tables\last update data\model_df_lean.parquet"
df_full.to_parquet(full_path, index=False)
df_lean.to_parquet(lean_path, index=False)
print(f"Saved model_df_full.parquet - {df_full.shape}")
print(f"Saved model_df_lean.parquet - {df_lean.shape}")
 
print("\nColumns in full but not lean:")
for c in sorted(set(df_full.columns) - set(df_lean.columns)):
    print(f"  {c}")

Saved model_df_full.parquet - (178384, 168)
Saved model_df_lean.parquet - (178384, 154)

Columns in full but not lean:
  clean_days_since_deep
  clean_days_since_deep_was_tracked
  clean_score_deep
  clean_score_deep_was_tracked
  pm_days_since_catering
  pm_days_since_catering_was_tracked
  pm_days_since_climate
  pm_days_since_climate_was_tracked
  pm_days_since_comms
  pm_days_since_comms_was_tracked
  pm_days_since_interior
  pm_days_since_interior_was_tracked
  pm_days_since_toilet
  pm_days_since_toilet_was_tracked
